In [ ]:
import pandas as pd
import numpy as np
import duckdb

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_folder = "staticNotDeployed/"
static_folder = "static/"

In [ ]:
# scenario_BNZ_path = "simulations_new/BNZ.csv"
# scenario_holder_path = "simulations_new/BNZ.csv"

# New data march 2025
scenario_BNZ_path = data_folder + "simulations_new/march2025/BNZ_?millions_annualy.csv"
scenario_holder_path = data_folder + "simulations_new/march2025/BNZ_?millions_annualy.csv"

socio_factors_path = "UK_Archetypes_global_measures.csv"

scenario_paths = [scenario_BNZ_path, scenario_holder_path]
scenario_names = ["BNZ", "Engagement", "Tailwinds", "Headwinds", "Innovation"]

## Create pandas tables

In [ ]:
df_socio = pd.read_csv(static_folder+socio_factors_path)

In [ ]:
# Remove one row on NaN
df_socio = df_socio.dropna()

In [ ]:
df_socio = df_socio.convert_dtypes()

In [ ]:
df_socio

In [ ]:
df_socio.rename(columns=lambda x: x.replace('.', '_'), inplace=True)

In [ ]:
df_socio.columns

In [ ]:
# EPC and gas_flag have weird values
df_socio.dtypes

In [ ]:
set(df_socio.EPC)

In [ ]:
set(df_socio.Gas_flag)

In [ ]:
# Replace the string 'unknown' with pd.NA
df_socio['Gas_flag'] = df_socio['Gas_flag'].replace('Y', pd.NA)

# Step 2: Convert to numeric (converts '1.0' to 1.0 as float)
df_socio['Gas_flag'] = pd.to_numeric(df_socio['Gas_flag'], errors='coerce')

df_socio['Gas_flag'] = df_socio['Gas_flag'].astype('Int16')

In [ ]:
# Replace the string 'unknown' with pd.NA
df_socio['EPC'] = df_socio['EPC'].replace('d', pd.NA)

# Step 2: Convert to numeric (converts '1.0' to 1.0 as float)
df_socio['EPC'] = pd.to_numeric(df_socio['EPC'], errors='coerce')

df_socio['EPC'] = df_socio['EPC'].astype('Int16')

### Explore BNZ Table

In [ ]:
df_BNZ = pd.read_csv(scenario_paths[0])

In [ ]:
df_BNZ = df_BNZ[~df_BNZ.isin(['#DIV/0!']).any(axis=1)]

In [ ]:
# Convert columns with names from 2025 to 2050 to float
year_columns = [str(year) for year in range(2025, 2051)]
df_BNZ[year_columns] = df_BNZ[year_columns].astype(np.float32)

In [ ]:
df_BNZ["total"] = df_BNZ[ [ f'{i}' for i in range(2025, 2051)]].sum(axis=1)

In [ ]:
df_BNZ[df_BNZ["Coben"] == "Total"].total

In [ ]:
df_BNZ[df_BNZ["Coben"] != "Total"].total

In [ ]:
np.sum(df_BNZ[df_BNZ["Coben"] == "Total"].total)

In [ ]:
np.sum(df_BNZ[df_BNZ["Coben"] != "Total"].total)

### Cobenefit table

In [ ]:
dfs = []
years = list(range( 2025, 2051 ))
years_col = [str(year) for year in years]

COBENEFS = ["Air quality", "Noise", "Excess cold", "Excess heat", "Dampness", "Congestion", "Hassle costs", "Road repairs", "Road safety", "Physical activity", "Diet change"]


# Function to add noise to each column based on its mean and std
def add_noise(column, year):
    # Apply noise on a year, given ONE cobenef at a time
    for cobenef in COBENEFS:
        values = column[column["Coben"] == cobenef][year]
        # print(values)
        mean = values.mean()
        std = values.std()

        # noise = np.random.normal(mean, std, size=values.shape)
        noise = np.random.normal(0, std, size=values.shape)
        
        # column[column["Coben"] == cobenef][year] = column[column["Coben"] == cobenef][year] + noise
        # column[column["Coben"] == cobenef][year] = column[column["Coben"] == cobenef][year]
        
        column.loc[column["Coben"] == cobenef, year] += noise

        # Function to add or subtract between 0% and 100% to each value
        def add_random_percent(value):
            change_factor = np.random.uniform(0, 1)  # random number between 0 and 1
            if np.random.choice([True, False]):  # randomly choose between addition or subtraction
                return value * (1 + change_factor)  # add between 0% and 100%
            else:
                return value * (1 - change_factor)  # subtract between 0% and 100%
        
        # Apply the function to the 'float_column'
        # Apply function only to 'col2' while leaving 'col1' unchanged
        # column.loc[column["Coben"] == cobenef, year] = df[['col1', 'col2']].apply(lambda x: add_random_percent(x) if x.name == 'col2' else x)
    
    return column
    

for i, scenario in enumerate(scenario_names):
    print(i)
    # if i == 2:
        # break
    # if i == 0:
        # continue

    # GENERATE FAKE DATA FOR NOW FOR ALL SCENATIOS EXCEPT BNZ
    if (scenario == "BNZ"):
        df_one_scenario = pd.read_csv(scenario_paths[i])
    else:
        df_one_scenario = pd.read_csv(scenario_paths[0])

        # Remove weird values
        df_one_scenario = df_one_scenario[~df_one_scenario.isin(['#DIV/0!']).any(axis=1)]
        
        # Convert columns with names from 2025 to 2050 to float
        df_one_scenario[years_col] = df_one_scenario[years_col].astype(np.float32)
        
        # Apply the function to each specified column
        for year in years:
            
            # df[str(year)] = df[str(year)](lambda col: add_noise(col), axis=0)
            # df[str(year)] = df.apply(lambda x: add_noise(x[str(year)]), axis=0)
            df_one_scenario[[str(year), "Coben"]] = add_noise(df_one_scenario[[str(year), "Coben"]], str(year))

            grouped_sum = df_one_scenario[df_one_scenario['Coben'] != 'Total'].groupby('Lookup Value')[str(year)].sum()
            # grouped_sum = df_one_scenario[df_one_scenario['Coben'] != 'Total'].groupby('Lookup Value')[str(year)].sum().reset_index()
            

            # print(grouped_sum)

            # Update the rows where 'Y' is 'total' with the corresponding sum
            df_one_scenario.loc[df_one_scenario['Coben'] == 'Total', str(year)] = df_one_scenario['Lookup Value'].map(grouped_sum)

            # Iterate through each group and assign the sum to rows where Y == 'total'
            # for _, row in grouped_sum.iterrows():
            #     lookup = row['Lookup Value']
            #     total_sum = row[str(year)]
            #     df_one_scenario.loc[(df_one_scenario['Lookup Value'] == lookup) & (df_one_scenario['Coben'] == 'Total'), str(year)] = total_sum

            # df_one_scenario.loc[df_one_scenario["Coben"] == "Total", ["Lookup Value", str(year)]] = grouped_sum  
        

    df_one_scenario["scenario"] = scenario_names[i]
    dfs.append(df_one_scenario)
    
df = pd.concat(dfs, axis=0)
del dfs 

In [ ]:
# Step: Drop rows where any column contains the value '#DIV/0!'
# There are rows with these weird values in the new dataset
df = df[~df.isin(['#DIV/0!']).any(axis=1)]

In [ ]:
df.dtypes

In [ ]:
# Convert columns with names from 2025 to 2050 to float
year_columns = [str(year) for year in range(2025, 2051)]
df[year_columns] = df[year_columns].astype(np.float32)

In [ ]:
df.dtypes

In [ ]:
# Create total column
# df["total (?m)"] = df[ [ f'{i} (?m)' for i in range(2025, 2051)]].sum(axis=1)
df["total (?m)"] = df[ [ f'{i}' for i in range(2025, 2051)]].sum(axis=1)

In [ ]:
# Delete all the columns per hh, only keep total value columns
# Not needed with new data
# df = df.drop(columns=[ f'{i} (?/hh)' for i in range(2025, 2051)])

In [ ]:
# Rename columns so it does not contain special characters (not needed anymore I think)
df.columns = df.columns.str.replace(' (?m)', '')

In [ ]:
# Rename columns: replace spaces with underscores
df.columns = df.columns.str.replace(' ', '_')

In [ ]:
df.head()

In [ ]:
# df.astype({'total': 'float'})

In [ ]:
df.total

In [ ]:
np.min(df.total)

In [ ]:
np.max(df.total)

In [ ]:
df.dtypes

## Join SE factors and cobenefs

In [ ]:
df = pd.merge(df, df_socio, left_on='Lookup_Value', right_on='LSOA_DZ_CD', how='inner')

In [ ]:
df.head()

In [ ]:
# Rename cobenef to always keep the same name (changed depending the version)
df.rename(columns={'Coben': 'co_benefit_type'}, inplace=True)

## Aggregating the time

In [ ]:
# Number of years merging
time_step = 5

In [ ]:
years = list(range( 2025, 2051 ))
#years

In [ ]:
df[["2025", "2026"]]

In [ ]:
#  AGGREGATE TIME: DISABLED FOR NOW
if False:
    for i in range(0, len(years) - ( time_step - 1), time_step):
        window_years = [str(year) for year in years[i:i+5]]
        print(window_years)
        window_sum = df[window_years].sum(axis=1)
        df[f'Y{window_years[0]}_{window_years[-1]}'] = window_sum

In [ ]:
# df[ ['2025', '2026', '2027', '2028', '2029', 'Y2025_2029'] ]

In [ ]:
# Delete single values columns for space
# df = df.drop(columns=[str(year) for year in years])

In [ ]:
# CONVERT INT64 into int32
df[df.select_dtypes(np.int64).columns] = df.select_dtypes(np.int64).astype(np.int32)

In [ ]:
df

In [ ]:
set(df.EPC)

In [ ]:
df.dtypes

In [ ]:
df[(df["co_benefit_type"] == "Noise") & (df["Lookup_Value"] == "N20000001")]

## Export table as parquet file

In [ ]:
df.to_parquet('static/database.parquet')

In [ ]:
# Not needed anymore, everything is in the same table
# df_socio.to_parquet('static/tableSocio.parquet')

## Export subset for faster development 

In [ ]:
df

In [ ]:
df.columns

In [ ]:
# Assuming df is your DataFrame and 'column_name' is the column you're interested in
# df_sampled = df.groupby('LAD').apply(lambda x: x.sample(n=10)).reset_index(drop=True)
df_sampled = df[df.Nation == "NI"]

In [ ]:
df_sampled

In [ ]:
df_sampled[(df_sampled["Lookup_Value"] == "N20000001") & (df_sampled["co_benefit_type"] == "Dampness")][["2028", "scenario"]]

In [ ]:
df_sampled.to_parquet('static/database_onlyIreland.parquet')

## Create Duckdb instance

The parquet file is currently used in the frontend as the .duckdb file was causing error. This part is therefore not useful for now

In [ ]:
DB_FILE_PATH = 'static/database.duckdb'
TABLE_NAME = "cobenefits"

In [ ]:
con = duckdb.connect(DB_FILE_PATH)

In [ ]:
# Create table and insert data
con.execute(f"DROP TABLE {TABLE_NAME}")

# Create table and insert data
con.execute(f"CREATE TABLE {TABLE_NAME} AS SELECT * FROM df")

# Verify data
result = con.execute(f"SELECT * FROM {TABLE_NAME} LIMIT 5").fetchall()
print("Sample data:")
print(result)

# Get and print schema
schema = con.execute(f"DESCRIBE {TABLE_NAME}").fetchall()
print("\nTable schema:")
for column in schema:
    print(f"{column[0]}: {column[1]}")

print(f"\nDatabase created and saved to: {DB_FILE_PATH}")